In [62]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Overwriting producer.py


In [69]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json
from datetime import datetime

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

historia_uzytkownikow = {}

for message in consumer:
    print(message.value)
    if message.value['amount'] > 1000:
        print("ALERT")
    
    user_id = message.value['user_id']
    now = datetime.fromisoformat(message.value['timestamp'])

    if user_id not in historia_uzytkownikow:
        historia_uzytkownikow[user_id] = []

    historia_uzytkownikow[user_id].append(now)
    nowa_lista = []
    for i in historia_uzytkownikow[user_id]:
        roznica = (now - i).total_seconds()
        if roznica <= 60:
            nowa_lista.append(i)
    
    historia_uzytkownikow[user_id] = nowa_lista
    ilosc_transakcji = len(historia_uzytkownikow[user_id])

    if ilosc_transakcji > 3:
        print(f"ALERT ILOŚCI TRANSAKCJI DLA {user_id}")

Overwriting consumer_filter.py
